In [16]:
# step 0 : creatinh the sample data
records = [  # Each dict = one Chroma row (id, text, metadata)
    {"id": "doc1", "text": "Customers can return products within 30 days of delivery.", "metadata": {"category": "returns"}},
    {"id": "doc2", "text": "Refunds are processed within 5 to 7 business days after the return is approved.", "metadata": {"category": "returns"}},
    {"id": "doc3", "text": "Orders above 499 rupees qualify for free shipping.", "metadata": {"category": "shipping"}},
    {"id": "doc4", "text": "You can reset your password from the account settings page.", "metadata": {"category": "account"}},
    {"id": "doc5", "text": "Express delivery orders usually arrive within 24 to 48 hours.", "metadata": {"category": "shipping"}},
]

# Note: Each dict is one record — like one SQL row.

In [17]:
# Step1 : Create the  client and the collection
import chromadb
from sentence_transformers import SentenceTransformer
from pprint import pprint

client=chromadb.PersistentClient("./Chroma_Store")

collection=client.get_or_create_collection(
    name="Support_Service",
    embedding_function=None
    )


In [18]:
# Checking if it made correctly or not 
print(collection.count())

0


In [19]:
documents=[]
ids=[]
metadata=[]

for i in records:
    documents.append(i["text"])
for i in records:
    ids.append(i["id"])
for i in records:
    metadata.append(i["metadata"])




In [20]:
model=SentenceTransformer("all-MiniLM-L6-v2")

embeddings=model.encode(
    documents,
    convert_to_numpy=True
).tolist()

collection.upsert(
    ids=ids,
    documents=documents,
    metadatas=metadata,
    embeddings=embeddings
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6807.43it/s]


In [21]:
# Checking if the upert is done correctly
print("Total Colection : ",collection.count())
print("Peek Sample :")
print(collection.peek())

Total Colection :  5
Peek Sample :
{'ids': ['doc1', 'doc2', 'doc3', 'doc4', 'doc5'], 'embeddings': array([[-0.04682997, -0.02426491,  0.0411784 , ..., -0.03994693,
         0.05715275, -0.02282485],
       [-0.06452115, -0.01745071,  0.08278479, ..., -0.04997708,
        -0.02753523, -0.01114142],
       [-0.03890188, -0.04008841,  0.04764163, ..., -0.0512668 ,
         0.02335656, -0.04045212],
       [-0.02485109, -0.05870529, -0.01641148, ...,  0.07588056,
        -0.01731195, -0.07811966],
       [ 0.01435259, -0.03327222,  0.05274634, ..., -0.00148862,
         0.06538489, -0.04434861]], shape=(5, 384)), 'documents': ['Customers can return products within 30 days of delivery.', 'Refunds are processed within 5 to 7 business days after the return is approved.', 'Orders above 499 rupees qualify for free shipping.', 'You can reset your password from the account settings page.', 'Express delivery orders usually arrive within 24 to 48 hours.'], 'uris': None, 'included': ['metadatas', 'd

In [30]:
# final step : queries
# --- Demo 1: The Returns Query ---
user_query_1 = "I want to return my shoes and get my money back"

query_embeddings_1=model.encode(
    user_query_1,convert_to_numpy=True
).tolist()

results_1=collection.query(
    query_embeddings=query_embeddings_1,
    n_results=1
)
print("Query:", user_query_1)
print("\nTop matches:")
for i in range(len(results_1["ids"][0])):
    print(f"Rank {i + 1}")
    print(" ID:", results_1["ids"][0][i])
    print(" Document:", results_1["documents"][0][i])
    print(" Distance:", results_1["distances"][0][i])
    print()

Query: I want to return my shoes and get my money back

Top matches:
Rank 1
 ID: doc2
 Document: Refunds are processed within 5 to 7 business days after the return is approved.
 Distance: 1.1705917119979858



In [31]:
# --- Demo 2: The Password Query ---
user_query_2 = "I have forgotten my login password?"

query_embedding_2 = model.encode([user_query_2], convert_to_numpy=True).tolist()

results_2 = collection.query(
    query_embeddings=query_embedding_2,
    n_results=1,  # Just grab the top 1 match this time
)

print("Query:", user_query_2)
print("\nTop Match:")
print(" ID:", results_2["ids"][0][0])
print(" Document:", results_2["documents"][0][0])

Query: I have forgotten my login password?

Top Match:
 ID: doc4
 Document: You can reset your password from the account settings page.
